## FINETUNE V2 FOR LEGAL DATA

In [2]:
import sentence_transformers

MODEL_DIR = ""
DB_DIR = ""


/home/nhminh/miniconda3/envs/DL_Env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import sqlite3

def get_data( batch = 10000):
        con = sqlite3.connect(DB_DIR)
        con.row_factory = sqlite3.Row
        query = """SELECT * FROM legal ORDER BY RANDOM() LIMIT 1000000"""
        cursor = con.cursor()
        cursor.execute(query)

        while True:
            rows = cursor.fetchmany(batch)
            if not rows:
                break

            for row in rows: 
                yield {
                    "anchor": "query: " + row["anchor"],
                    "positive": "passage: " + row["positive"],
                }


In [ ]:
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainingArguments, SentenceTransformerTrainer)
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from datasets import Dataset

model = SentenceTransformer(MODEL_DIR)
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir="models/vietnamese-embedding",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=100,

    save_strategy="steps",
    save_steps=5000,
    save_total_limit=2,
)

train_dataset = Dataset.from_generator(get_data)

trainer = SentenceTransformerTrainer(
    model=model, 
    args=args,
    train_dataset=train_dataset,
    loss=loss
)

/tmp/ipykernel_45434/2688949519.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments, losses


AttributeError: 'NoneType' object has no attribute 'parameters'

In [ ]:
trainer.train()